# Insight 2: Vote Buying Visualization
วิเคราะห์ความสัมพันธ์ระหว่างคะแนน ส.ส. เขตและบัญชีรายชื่อ เพื่อหาความผิดปกติ (Outliers) ตามเงื่อนไขและเพิ่มเติมเทคนิคการหา Outlier ทางสถิติด้วย Z-Score

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import zscore

# โหลดข้อมูลที่รวมแล้ว
df = pd.read_csv("merged_parties.csv")

# 1. คำนวณ ratio = คะแนน partylist / (คะแนนสส.เขต + 1)
df["ratio"] = df["list_votes"] / (df["district_votes"] + 1)

# ตรวจสอบข้อมูลเบื้องต้น
display(df.head())

,party_name,district_votes,list_votes,ratio
0,0001-ไทยทรัพย์ทวี,2646,305566,115.438610
1,0002-เพื่อชาติไทย,538,680256,1262.070501
2,0003-ใหม่,892,314430,352.105263
3,0004-มิติใหม่,1310,247367,188.685736
4,0005-รวมใจไทย,2956,435225,147.184647


In [2]:
# 2. ดึงหมายเลขพรรคเพื่อแบ่งเป็นพรรคต้นๆ กับพรรคอื่นๆ
# ดึงตัวเลขนำหน้าตัวแรกจาก party_name (เช่น 0001 -> 1)
df["party_number"] = df["party_name"].str.extract(r"^(\d+)").astype(float)

# กำหนด "เลขพรรคต้นๆ" (เช่น หมายเลข 1 ถึง 20) ผู้ใช้สามารถปรับแก้ตัวเลขนี้ได้ตามต้องการ
EARLY_PARTY_THRESHOLD = 20


In [3]:
# 3. ใช้สถิติ Z-Score เพื่อหาจุดที่ Outline จริงๆ (ตามภาพแนะนำ)
# หา Z-Score ของ ratio
df["z_score"] = zscore(df["ratio"])

# ให้จุดที่เป็น Outlier มี Z-Score > 3
df["is_outlier"] = df["z_score"] >= 3

# จัดกลุ่ม (Category) สีเพื่อนำไป Plot
def assign_category(row):
    if row["is_outlier"]:
        return "Outlier ทางสถิติ (แดง)"
    elif row["party_number"] <= EARLY_PARTY_THRESHOLD:
        return "เลขพรรคต้นๆ (เขียว)"
    else:
        return "พรรคอื่นๆ (เทา)"

df["category"] = df.apply(assign_category, axis=1)

# บันทึกข้อมูลที่คำนวณแล้วกลับลงไปใน CSV อีกรอบ
df.to_csv("merged_parties_with_ratio.csv", index=False, encoding="utf-8-sig")
print("บันทึกไฟล์ที่มี ratio และ z_score ลง merged_parties_with_ratio.csv แล้ว")

บันทึกไฟล์ที่มี ratio และ z_score ลง merged_parties_with_ratio.csv แล้ว


In [4]:
# 4. วาดกราฟ Scatter Plot + ระบบ Interactive แบบ Hover ดูข้อมูลได้
color_discrete_map = {
    "Outlier ทางสถิติ (แดง)": "red",
    "เลขพรรคต้นๆ (เขียว)": "green",
    "พรรคอื่นๆ (เทา)": "gray"
}

fig = px.scatter(
    df, 
    x="district_votes", 
    y="list_votes", 
    color="category",
    color_discrete_map=color_discrete_map,
    hover_name="party_name",
    hover_data={
        "district_votes": True,
        "list_votes": True,
        "ratio": ":.2f",
        "z_score": ":.2f",
        "party_number": False,
        "category": False,
        "is_outlier": False
    },
    title="วิเคราะห์เปรียบเทียบคะแนน ส.ส. เขต กับ บัญชีรายชื่อ",
    labels={
        "district_votes": "คะแนนเขต (District Votes)",
        "list_votes": "คะแนนบัญชีรายชื่อ (Party-List Votes)",
    }
)

# วาดเส้น y = x (Baseline)
max_val = max(df["district_votes"].max(), df["list_votes"].max())
fig.add_shape(
    type="line", 
    x0=0, y0=0, 
    x1=max_val, y1=max_val,
    line=dict(color="black", dash="dash"),
    name="y=x (Baseline)"
)

# เพิ่ม annotation ให้เส้น y=x
fig.add_annotation(
    x=max_val * 0.5,
    y=max_val * 0.5,
    text="Baseline (y = x)",
    showarrow=True,
    arrowhead=2,
    ax=-40,
    ay=-40
)

# ตั้งค่าจุดให้ใหญ่ขึ้นเล็กน้อย
fig.update_traces(marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")))

fig.show()